# 언어 모델 평가의 첫걸음: 펄플렉서티(PPL)

언어 모델이 문장을 얼마나 '잘' 생성하는지 평가할 때 가장 먼저 마주하게 되는 지표가 바로 **펄플렉서티(Perplexity, PPL)** 입니다. 복잡한 수식 이전에, 우리 뇌가 언어를 이해하는 방식에 빗대어 PPL을  파악해 보겠습니다.

## 1. "얼마나 당황스러운가?" - 헷갈림의 정도

'Perplexity'라는 단어의 사전적 의미는 **'당혹감'** 또는 **'복잡함'** 입니다. 언어 모델 입장에서 생각하면, 다음에 올 단어를 예측할 때 "얼마나 갈팡질팡하고 있는가"를 수치로 나타낸 것입니다.

- **낮은 PPL**: 모델이 다음에 올 단어를 명확하게 예측하고 있음 (확신이 높음)
- **높은 PPL**: 모델이 어떤 단어가 올지 몰라 당황하고 있음 (불확실성이 높음)


## 2. 분기 계수(Branching Factor): 선택지의 개수

PPL 수치를 가장 쉽게 이해하는 방법은 **'선택지의 평균 개수'** 로 생각하는 것입니다.

예를 들어, 어떤 모델의 PPL이 **10** 이 나왔다면 이는 다음과 같은 의미입니다.
> "이 모델은 매 단계마다 평균적으로 **10개의 선택지** 중에서 어떤 것이 정답인지 고민하고 있다

### 객관식 시험 비유
- **PPL = 2**: 마치 2지선다형(O/X) 문제를 푸는 것과 같습니다. 정답을 맞히기 매우 쉽고 명확합니다.
- **PPL = 100**: 마치 100지선다형 문제를 푸는 것과 같습니다. 정답을 맞히기가 훨씬 어렵고 모델은 매우 혼란스러운 상태입니다.

따라서 좋은 언어 모델일수록 문맥을 날카롭게 파악하여 이 '선택지의 개수(분기 계수)'를 1에 가깝게 줄여나갑니다.

## 3. 문맥(Context)의 힘

똑같은 단어라도 문맥에 따라 PPL은 달라집니다.

*   **문맥이 없는 경우**: "나는 [ ? ]"
    *   '밥을', '학교에', '너를', '어제' 등 수많은 단어가 올 수 있어 PPL이 매우 높습니다.
*   **문맥이 풍부한 경우**: "배가 너무 고파서 식당에 가자마자 [ ? ]"
    *   '주문을', '밥을' 등 몇 가지 단어로 압축됩니다. PPL이 급격히 낮아집니다.

숙련된 모델일수록 앞선 단어들을 통해 후보군을 좁히는 능력이 탁월하며, 결과적으로 전체 문장에 대한 PPL을 낮게 유지합니다.

## 요약
- PPL은 모델의 **불확실성** 을 측정하는 지표입니다.
- **분기 계수** 관점에서 보면, 모델이 고민하는 평균 선택지 개수를 의미합니다.
- **수치가 낮을수록** 모델의 예측 성능이 뛰어나고 문맥 파악 능력이 좋다는 것을 뜻합니다.

# 펄플렉서티(PPL)의 수학적 기초와 로그 확률의 필요성

펄플렉서티(Perplexity)는 언어 모델의 성능을 확률론적 관점에서 정밀하게 측정하는 지표입니다. 이 문서에서는 PPL의 수식적 정의와 실제 계산 과정에서 발생하는 수치적 문제, 그리고 이를 해결하기 위한 로그 연산의 도입 과정을 상세히 다룹니다.

## 1. PPL의 수학적 정의

언어 모델은 문장 $W = (w_1, w_2, ..., w_N)$이 발생할 확률 $P(W)$를 계산합니다. PPL은 이 문장 확률의 역수에 대해 문장 길이 $N$만큼 기하평균을 취한 값으로 정의됩니다.

$$PPL(W) = P(w_1, w_2, ..., w_N)^{-\frac{1}{N}} = \sqrt[N]{\frac{1}{P(w_1, w_2, ..., w_N)}}$$

체인 룰(Chain Rule)을 적용하여 문장의 결합 확률을 조건부 확률의 곱으로 나타내면 다음과 같습니다.

$$PPL(W) = \sqrt[N]{\frac{1}{\prod_{i=1}^{N} P(w_i | w_1, ..., w_{i-1})}}$$

## 2. 컴퓨터 공학적 한계: 언더플로우(Underflow)

수식 그대로 PPL을 계산하려면 0과 1 사이의 작은 확률값들을 수십, 수백 번 곱해야 합니다.

*   예: 단어 하나의 확률이 0.1일 때, 100개 단어의 문장 확률은 $0.1^{100} = 10^{-100}$이 됩니다.
*   **문제점**: 현대 컴퓨터의 부동소수점(Floating Point) 표현 한계를 넘어서면 값이 **0으로 증발(Underflow)** 해버립니다. 0으로 나누기는 불가능하므로 PPL 계산 자체가 실패하게 됩니다.

## 3. 로그(Log) 함수의 도입과 수식 변형

이 문제를 해결하기 위해 **로그(Log)** 를 사용합니다. 로그의 성질 ($\log(a \times b) = \log a + \log b$)을 이용하면 확률의 곱셈을 로그 확률의 덧셈으로 변환할 수 있습니다.

PPL 수식에 자연로그($\ln$)를 취하고 지수함수($\exp$)로 되돌리는 과정을 거치면 다음과 같은 '안전한' 수식이 완성됩니다.

$$PPL(W) = \exp \left( -\frac{1}{N} \sum_{i=1}^{N} \log P(w_i | w_{1...i-1}) \right)$$

이 수식은 로그 공간에서 덧셈을 수행하므로 언더플로우 위협에서 자유롭습니다.

## 4. 수치적 증명 (Numerical Proof)

모델이 2개의 단어로 이루어진 문장($N=2$)에서 각각의 발생 확률을 **0.1**로 예측했다고 가정해 봅시다.

### A. 일반 수식 계산
1.  문장 확률: $P(W) = 0.1 \times 0.1 = 0.01$
2.  PPL 계산: $PPL = (1 / 0.01)^{1/2} = 100^{0.5} = \mathbf{10}$

### B. 로그 수식 계산 (자연로그 사용)
1.  각 단어의 로그 확률: $\log(0.1) \approx -2.3026$
2.  로그 확률의 합: $-2.3026 + (-2.3026) = -4.6052$
3.  평균 및 부호 반전: $-\frac{1}{2} \times (-4.6052) = 2.3026$
4.  지수 함수 환원: $\exp(2.3026) \approx \mathbf{10}$

**결과**: 두 방식의 결과가 일치함을 알 수 있습니다.